# 📚 RAG-Based Educational Chatbot
### Assignment: Building a Simple RAG-Based Educational Chatbot

This notebook implements a **Retrieval-Augmented Generation (RAG)** chatbot for school students covering:
- Mathematics · Science · English · Social Studies

**Tasks covered:**
| Task | Description |
|------|-------------|
| Task 1 | RAG Pipeline Design |
| Task 2 | Chunking Strategy |
| Task 3 | Embedding Model + Vector Database |
| Task 4 | Prompt Engineering |
| Task 5 | Build & Run the Chatbot |

---


## Task 1 — RAG Pipeline Design

The complete pipeline works in two phases:

**Indexing Phase** (run once)
```
Documents → Text Chunking → Embedding Model → FAISS Vector Store
```

**Querying Phase** (every student question)
```
Student Question → Embed Query → Similarity Search → Top-K Chunks
                                                          ↓
                                                  Build Prompt
                                                          ↓
                                                  Claude LLM → Answer
```


## Step 0 — Install Dependencies

In [ ]:
# Install required libraries
!pip install sentence-transformers faiss-cpu openai -q
print("✅ All packages installed!")

## Step 1 — Imports & Setup

In [ ]:
import os
import re
import textwrap
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI

print("✅ All libraries imported successfully!")

## Step 2 — Set Your OpenAI API Key

Get your free API key from: https://platform.openai.com

Paste it below 👇

In [ ]:
from google.colab import userdata

# Option A: Store in Colab Secrets (recommended — click the 🔑 key icon in the left sidebar)
# Add a secret named OPEN_API_KEY with your key value, then run this cell.
try:
    OPEN_API_KEY = userdata.get('OPEN_API_KEY')
    print("✅ API key loaded from Colab Secrets!")
except Exception:
    # Option B: Paste directly (less secure)
    OPEN_API_KEY = "sk-..."   # ← Replace with your OpenAI key
    print("⚠️  Using hardcoded key. Consider using Colab Secrets instead.")

os.environ["OPEN_API_KEY"] = OPEN_API_KEY

## Task 2 — Chunking Strategy

**Chosen Strategy:** Sentence-aware fixed-size chunking

| Parameter | Value | Reason |
|-----------|-------|--------|
| Chunk size | 300 words | Covers a full textbook concept without being too long |
| Overlap | 50 words | Preserves context at chunk boundaries |
| Split on | Sentence boundaries | Avoids cutting mid-concept |

**Why this works for textbooks:**
- Textbooks present one concept per paragraph
- Sentence-level splits keep complete ideas together
- Overlap prevents losing context between retrieved chunks
- Fixed size keeps retrieval scores consistent and comparable


In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list:
    """
    Sentence-aware fixed-size chunking.

    Strategy:
    - Split text into sentences using punctuation rules
    - Group sentences into chunks of ~chunk_size words
    - Keep last `overlap` words from previous chunk for context continuity
    """
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current_words, current_chunk = [], 0, []

    for sentence in sentences:
        word_count = len(sentence.split())
        if current_words + word_count > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))
            overlap_words = " ".join(current_chunk).split()[-overlap:]
            current_chunk = list(overlap_words) + sentence.split()
            current_words = len(current_chunk)
        else:
            current_chunk.extend(sentence.split())
            current_words += word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

# Test the chunker
test_text = """Photosynthesis is the process by which plants make food.
Plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.
This process takes place in chloroplasts. Chlorophyll absorbs sunlight.
The chemical equation is: 6CO2 + 6H2O + light → C6H12O6 + 6O2."""

test_chunks = chunk_text(test_text, chunk_size=30, overlap=5)
print(f"Sample text split into {len(test_chunks)} chunk(s):")
for i, c in enumerate(test_chunks):
    print(f"  Chunk {i+1}: {c[:80]}...")

## Task 3 — Embedding Model & Vector Database

**Embedding Model:** `all-MiniLM-L6-v2` (via sentence-transformers)
- Lightweight (22 MB), fast inference
- 384-dimensional embeddings
- Trained on 1 billion sentence pairs — excellent at semantic Q&A
- Free, no API key needed

**Vector Database:** FAISS (Facebook AI Similarity Search)
- Runs fully in memory — no server needed
- `IndexFlatL2` gives exact nearest-neighbor search
- Perfect for small-to-medium educational document sets
- Scales to IVF/HNSW for larger corpora


In [ ]:
class VectorStore:
    """
    Manages document embeddings and similarity search using FAISS.
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"⏳ Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.chunks = []
        print("✅ Embedding model loaded!")

    def add_documents(self, chunks: list):
        """Embed chunks and store in FAISS index."""
        self.chunks = chunks
        print(f"⚙️  Embedding {len(chunks)} chunks...")
        embeddings = self.model.encode(chunks, show_progress_bar=True)
        embeddings = np.array(embeddings, dtype="float32")
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)
        print(f"✅ {len(chunks)} chunks stored in FAISS (dim={dim})")

    def search(self, query: str, top_k: int = 3) -> list:
        """Find the top_k most relevant chunks for a query."""
        q_vec = self.model.encode([query], show_progress_bar=False)
        q_vec = np.array(q_vec, dtype="float32")
        distances, indices = self.index.search(q_vec, top_k)
        return [self.chunks[i] for i in indices[0] if i < len(self.chunks)]

# Initialise the vector store
vector_store = VectorStore()

## Task 4 — Prompt Engineering

**Design Decisions:**

| Element | Choice | Why |
|---------|--------|-----|
| Role framing | "friendly and patient teacher" | Sets student-appropriate tone |
| Context block | Clearly delimited with `────` | LLM knows exactly what facts to use |
| Strict grounding | "Use ONLY the context below" | Prevents hallucination |
| Fallback | Explicit "I don't know" instruction | Honest over fabricating |
| Style guide | Simple language, step-by-step | Makes answers student-friendly |


In [ ]:
PROMPT_TEMPLATE = """You are a friendly and patient teacher helping school students understand their subjects.
Use ONLY the information provided in the context below to answer the student's question.
If the answer is not found in the context, say: "I don't have enough information on that topic in your study materials."

Rules:
- Use simple, clear language suitable for school students.
- Break down complex ideas into easy steps when needed.
- Keep answers concise (3–6 sentences unless more detail is truly needed).
- Do NOT make up facts. Stick strictly to the provided context.

────────────────────────────────────────
CONTEXT FROM STUDY MATERIALS:
{context}
────────────────────────────────────────

STUDENT QUESTION: {question}

ANSWER:"""

def build_prompt(question: str, context_chunks: list) -> str:
    """Assemble the final prompt from retrieved chunks and the student question."""
    context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
    return PROMPT_TEMPLATE.format(context=context, question=question)

# Preview the prompt template
print(PROMPT_TEMPLATE[:300] + "\n... [rest of template]")

## Task 5 — Build the Chatbot

### 5a. Load Educational Documents

You can replace the sample documents below with your own textbook content.  
To use a PDF from Google Drive, see the cell after this one.


In [ ]:
# ── Sample educational documents (inline) ────────────────────────────────
SAMPLE_DOCUMENTS = {
    "science_photosynthesis.txt": """
    Photosynthesis is the process by which green plants make their own food.
    Plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.
    This process takes place in the chloroplasts, which contain a green pigment called chlorophyll.
    Chlorophyll absorbs sunlight and uses its energy to convert carbon dioxide and water into glucose.
    The chemical equation for photosynthesis is: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2.
    Oxygen is released as a by-product during this process, which is essential for animal life.
    Photosynthesis is the foundation of almost all food chains on Earth.
    Without photosynthesis, there would be no oxygen in the atmosphere and no food for animals.
    """,

    "math_fractions.txt": """
    Fractions represent parts of a whole. A fraction has two parts: the numerator and the denominator.
    The numerator is the number above the line and shows how many parts we have.
    The denominator is the number below the line and shows total equal parts the whole is divided into.
    To add fractions with the same denominator, simply add the numerators and keep the denominator.
    For example, 1/4 + 2/4 = 3/4.
    To add fractions with different denominators, first find the Least Common Denominator (LCD).
    Convert each fraction to an equivalent fraction with the LCD, then add.
    For example, 1/3 + 1/4: LCD is 12, so 4/12 + 3/12 = 7/12.
    Simplify the result when possible by dividing numerator and denominator by their GCF.
    """,

    "english_grammar.txt": """
    A noun is a word that names a person, place, thing, or idea.
    Examples of nouns: teacher, city, book, freedom.
    Nouns can be singular (one) or plural (more than one).
    Most nouns become plural by adding -s or -es. For example: cat → cats, box → boxes.
    Proper nouns name specific people or places and always begin with a capital letter.
    Common nouns refer to general things and are not capitalized unless at the start of a sentence.
    A pronoun replaces a noun to avoid repetition. Examples: he, she, it, they, we, I.
    Verbs are action words that describe what a subject does. Examples: run, think, write, is.
    Adjectives describe or modify nouns, answering questions like what kind, how many, or which one.
    Adverbs modify verbs, adjectives, or other adverbs and often end in -ly.
    """,

    "social_studies_water_cycle.txt": """
    The water cycle, also called the hydrological cycle, describes how water moves continuously
    through the environment. The four main stages are evaporation, condensation, precipitation,
    and collection.
    Evaporation occurs when heat from the sun turns liquid water from oceans, rivers, and lakes
    into water vapor that rises into the atmosphere.
    Condensation happens when water vapor cools and forms tiny water droplets that create clouds.
    Precipitation occurs when water droplets in clouds combine and fall as rain, snow, sleet, or hail.
    Collection refers to water gathering in oceans, rivers, lakes, and underground aquifers,
    completing the cycle. The water cycle is driven by solar energy and gravity.
    It is essential for distributing fresh water across the Earth and supporting all living things.
    """,
}

print(f"📄 Loaded {len(SAMPLE_DOCUMENTS)} documents:")
for name in SAMPLE_DOCUMENTS:
    word_count = len(SAMPLE_DOCUMENTS[name].split())
    print(f"   • {name} ({word_count} words)")

### 5b. (Optional) Load Your Own PDF from Google Drive

In [ ]:
# Uncomment and run this cell to load your own PDF textbook from Google Drive
# Requires: pip install PyPDF2

# from google.colab import drive
# import PyPDF2
#
# drive.mount('/content/drive')
# PDF_PATH = '/content/drive/MyDrive/your_textbook.pdf'  # ← Change this path
#
# def load_pdf(path):
#     text = ""
#     with open(path, 'rb') as f:
#         reader = PyPDF2.PdfReader(f)
#         for page in reader.pages:
#             text += page.extract_text() + "\n"
#     return text
#
# pdf_text = load_pdf(PDF_PATH)
# SAMPLE_DOCUMENTS['my_textbook.pdf'] = pdf_text
# print(f"✅ Loaded PDF: {len(pdf_text.split())} words")

print("ℹ️  PDF loading is optional. Using sample documents for now.")

### 5c. Chunk and Index All Documents

In [ ]:
# Chunk all documents and build the FAISS index
all_chunks = []
chunk_sources = []  # Track which document each chunk came from

for doc_name, doc_text in SAMPLE_DOCUMENTS.items():
    chunks = chunk_text(doc_text, chunk_size=300, overlap=50)
    print(f"   {doc_name}: {len(chunks)} chunks")
    all_chunks.extend(chunks)
    chunk_sources.extend([doc_name] * len(chunks))

print(f"\n📦 Total chunks: {len(all_chunks)}")

# Store in FAISS
vector_store.add_documents(all_chunks)

### 5d. LLM Response Function (Claude API)

In [ ]:
def get_answer(question: str, top_k: int = 3, verbose: bool = False) -> str:
    """
    Full RAG pipeline:
    1. Embed the question
    2. Retrieve top_k relevant chunks from FAISS
    3. Build the prompt
    4. Call OpenAI GPT and return the answer
    """
    # Step 1: Retrieve
    relevant_chunks = vector_store.search(question, top_k=top_k)

    if verbose:
        print("🔍 Retrieved chunks:")
        for i, chunk in enumerate(relevant_chunks):
            print(f"  [{i+1}] {chunk[:100]}...")
        print()

    # Step 2: Build prompt
    prompt = build_prompt(question, relevant_chunks)

    # Step 3: Call OpenAI
    client = OpenAI(api_key=OPEN_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

# Quick smoke test
print("🧪 Testing the pipeline...")
test_answer = get_answer("What is photosynthesis?", verbose=True)
print("🤖 Answer:")
print(textwrap.fill(test_answer, width=80))

### 5e. Interactive Chatbot

Run the cell below to start chatting with the educational assistant.  
Type your question and press **Enter**. Type `quit` to stop.


In [ ]:
print("=" * 60)
print("  📚 RAG Educational Chatbot — School Assistant")
print("=" * 60)
print("Ask questions about: Science | Math | English | Social Studies")
print("Type 'quit' to exit.\n")

while True:
    question = input("🧑‍🎓 You: ").strip()

    if not question:
        continue
    if question.lower() in {"quit", "exit", "bye", "q"}:
        print("👋 Goodbye! Keep learning!")
        break

    print("\n🤖 Chatbot: ", end="")
    try:
        answer = get_answer(question, top_k=3)
        # Nicely wrap long answers
        wrapped = textwrap.fill(answer, width=72)
        print(wrapped)
    except Exception as e:
        print(f"[Error: {e}]")
    print()

### 5f. Batch Test — Try Sample Questions

In [ ]:
sample_questions = [
    "What is photosynthesis and where does it happen?",
    "How do you add fractions with different denominators?",
    "What is the difference between a noun and a pronoun?",
    "What are the four stages of the water cycle?",
    "What is the capital of France?",   # Not in documents — should say it doesn't know
]

print("=" * 65)
for q in sample_questions:
    print(f"\n🧑‍🎓 Q: {q}")
    answer = get_answer(q, top_k=3)
    print(f"🤖 A: {textwrap.fill(answer, width=60, subsequent_indent='     ')}")
    print("-" * 65)

---
## Summary

| Task | Implementation |
|------|---------------|
| **Task 1 — RAG Pipeline** | Document loading → Chunking → Embedding → FAISS → Prompt → Claude |
| **Task 2 — Chunking** | Sentence-aware fixed-size (300 words, 50-word overlap) |
| **Task 3 — Embedding + VDB** | `all-MiniLM-L6-v2` + FAISS `IndexFlatL2` |
| **Task 4 — Prompt Engineering** | Role-framed, context-grounded, anti-hallucination template |
| **Task 5 — Chatbot** | Interactive CLI loop + batch testing |

**To extend this project:**
- Add more subject documents (load from PDF or text files)
- Switch to `IndexIVFFlat` in FAISS for larger document sets
- Add a Gradio UI: `pip install gradio` and wrap `get_answer()` in `gr.Interface`
- Stream responses using `client.messages.stream()`
